# PFAS — HGT + XGBoost hybride **GPU / config CORRIGÉE anti-sur-lissage** (EPA 2024)

Version **07** : corrige le notebook 06, dont la « config maximale » avait *dégradé* les
modèles à base de graphe (Fusion AUC 0.916 → 0.879) à cause d'un **graphe trop dense**
(k=15 spatial + k=15 features ≈ 5× d'arêtes → **sur-lissage du GNN** + **explosion RAM**).

Objectif : **retrouver / dépasser la Fusion 0.916** du notebook 05 tout en **tenant en RAM**.

## Corrections vs notebook 06
| Levier | 06 (dégradé) | **07 (corrigé)** |
|---|---|---|
| KNN spatial `sample↔sample` | k=15 | **k=8** |
| KNN espace des features | k=15 | **désactivé** (`KNN_FEATURE_K=0`) ← cause RAM + sur-lissage |
| Architecture HGT | empilée simple | **résiduel + LayerNorm** (anti-sur-lissage) |
| `num_layers` (Optuna) | 1-3 | **1-2** |
| `hidden` / `heads` (Optuna) | {64,128,256}/{2,4,8} | **{64,128}/{2,4}** (RAM) |
| Epochs HGT | 300 (plateau dès ~120) | **200 + early-stopping** (patience 30) |

Conservé : **split par puits** (pas de fuite), **48 features**, **sans SMOTE**
(`scale_pos_weight`), 4 modèles de base (HGT, XGB, LGBM, **CatBoost**), **stacking OOF**,
**seuil optimisé** + **calibration isotonique** (Brier).

## Instructions
1. `pip install torch torch_geometric xgboost lightgbm catboost optuna scikit-learn pyarrow shap`.
2. Placer `CA-PFAS-ASGWS.parquet` (ou `.csv`) dans `DATA_DIR`.
3. **Run all**. Renvoyer `outputs/hgt_hybrid_v2/results_v2.csv` et `summary_v2.json`.

> Bien plus léger que le 06 (graphe creux + early-stopping) : ≈ 30-50 min sur 1 GPU T4.
> Si la RAM reste juste : baisser `KNN_SPATIAL_K` à 6, `HGT_OPTUNA_TRIALS`/`N_TRIALS`.

## 1. Imports + configuration (régler ici)

In [ ]:
import warnings, json, pickle, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv, Linear
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (classification_report, roc_auc_score, f1_score, accuracy_score,
    precision_score, recall_score, brier_score_loss, roc_curve, average_precision_score,
    precision_recall_curve, ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve
import xgboost as xgb, lightgbm as lgb, optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
try:
    from catboost import CatBoostClassifier; CATBOOST_AVAILABLE = True
except Exception:
    CATBOOST_AVAILABLE = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU = device.type == 'cuda'
np.random.seed(42); torch.manual_seed(42)
if GPU: torch.cuda.manual_seed_all(42)

# ------------------- RÉGLAGES (config CORRIGÉE anti-sur-lissage) -------------------
DATA_DIR            = Path('../data/processed')      # dossier contenant CA-PFAS-ASGWS.parquet / .csv
OUT_FIG             = Path('../reports/figures/hgt_hybrid_v2')
OUT_RES             = Path('../outputs/hgt_hybrid_v2')
OUT_MODELS          = Path('../models')
EPOCHS_HGT          = 200
EARLY_STOP_PATIENCE = 30      # arrêt si pas d'amélioration Val F1 (le 06 plafonnait dès ~120)
RUN_OPTUNA_HGT      = True
HGT_OPTUNA_TRIALS   = 20
HGT_OPTUNA_EPOCHS   = 80      # epochs/essai (avec early-stopping) ; final ré-entraîné à EPOCHS_HGT
N_TRIALS            = 40      # Optuna fusion + stacking
KNN_SPATIAL_K       = 8       # graphe CREUX (anti sur-lissage + RAM)
KNN_FEATURE_K       = 0       # 0 = feature-KNN DÉSACTIVÉ (cause RAM + sur-lissage en 06) ; 5 max si réactivé
RESIDUAL_HGT        = True    # connexions résiduelles + LayerNorm dans le HGT
ADD_CATBOOST        = CATBOOST_AVAILABLE
SPATIAL_BLOCKING    = False   # True = split groupé par county (test inter-régions, plus strict)
HIDDEN_DEFAULT      = 128     # si RUN_OPTUNA_HGT=False
XGB_DEVICE          = 'cuda' if GPU else 'cpu'
CB_TASK             = 'GPU' if GPU else 'CPU'
# ----------------------------------------------------------------------------------
for d in (OUT_FIG, OUT_RES, OUT_MODELS): d.mkdir(parents=True, exist_ok=True)
print(f'device = {device} | GPU = {GPU} | CatBoost = {CATBOOST_AVAILABLE} | torch {torch.__version__}')

## 2. Chargement + cible EPA 2024 (identique aux notebooks de référence)

In [ ]:
parquet = DATA_DIR/'CA-PFAS-ASGWS.parquet'; csv = DATA_DIR/'CA-PFAS-ASGWS.csv'
try:
    df = pd.read_parquet(parquet) if parquet.exists() else pd.read_csv(csv)
except Exception as _e:           # pyarrow/fastparquet absent -> CSV
    print('parquet indisponible, lecture CSV:', _e); df = pd.read_csv(csv)
EPA2024_MCLS = {'PFOA_ngL':4.0,'PFOS_ngL':4.0,'PFNA_ngL':10.0,'PFHxS_ngL':10.0,'HFPO_DA_ngL':10.0}
HI_REFS      = {'PFNA_ngL':10.0,'PFHxS_ngL':10.0,'HFPO_DA_ngL':10.0,'PFBS_ngL':2000.0}
def compute_target_epa2024(d):
    ex = pd.Series(False, index=d.index)
    for c,m in EPA2024_MCLS.items():
        if c in d.columns: ex |= d[c].fillna(0) > m
    ex |= sum(d[c].fillna(0)/r for c,r in HI_REFS.items() if c in d.columns) > 1.0
    return ex.astype(int)
df['is_contaminated'] = compute_target_epa2024(df)
print(f'{df.shape[0]:,} lignes x {df.shape[1]} cols | puits uniques: {df.gm_well_id.nunique():,}')
print(f"pos: {df.is_contaminated.mean()*100:.1f}%")
counts = df['is_contaminated'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6,4))
b = ax.bar(['Non-contaminated (0)','Contaminated (1)'], counts.values, color=['#3498db','#e74c3c'], edgecolor='black')
for bb,v in zip(b, counts.values): ax.text(bb.get_x()+bb.get_width()/2., bb.get_height(), f'{v}\n({100*v/len(df):.1f}%)', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Number of samples'); plt.tight_layout(); plt.savefig(OUT_FIG/'fig1_class_distribution.png', dpi=300, bbox_inches='tight'); plt.show()

## 3. Features enrichies (48) + 3 catégoriels encodés

In [ ]:
df['sand_silt_ratio']      = df['soil_sand_pct'] / (df['soil_silt_pct'] + 1e-6)
df['geotracker_proximity'] = 1.0 / (1.0 + df['dist_geotracker_km'])
dt = pd.to_datetime(df['collection_date'])
df['year'] = dt.dt.year; df['month'] = dt.dt.month; df['season'] = ((dt.dt.month % 12)//3 + 1).astype(int)
CAT = ['gm_well_category','nearest_geotracker_type','county']
df[[c+'_enc' for c in CAT]] = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1).fit_transform(df[CAT].astype(str))
FEATURE_CATEGORIES = {
 'Environmental': ['rainfall_mm_month','runoff_mm','et_mm_month','soil_moisture_total_mm','root_zone_moist_kg_m2',
   'temp_c','snowpack_mm','soil_sand_pct','soil_silt_pct','soil_clay_pct','soil_om_pct','soil_ph','soil_ksat_um_s',
   'soil_awc_cm_cm','soil_bulk_density','sand_silt_ratio'],
 'Facility_Proximity': ['dist_geotracker_km','n_geotracker_within_1km','n_geotracker_within_3km','n_geotracker_within_10km','n_geotracker_within_50km','geotracker_proximity'],
 'Water_Quality': ['cocontam_tds','cocontam_no3n','cocontam_so4','cocontam_tce','cocontam_pce','cocontam_mtbe','cocontam_as','cocontam_mn','cocontam_fe','cocontam_bz'],
 'Air_Quality': ['aqs_pm25_ugm3','aqs_pm10_ugm3','aqs_no2_ppb','aqs_so2_ppb','aqs_ozone_ppb','aqs_co_ppm','aqs_wind_ms','aqs_humidity_pct'],
 'Temporal': ['year','month','season'],
 'Context': [c+'_enc' for c in CAT],
 'Geospatial': ['latitude','longitude'],
}
all_feature_cols = []
for v in FEATURE_CATEGORIES.values(): all_feature_cols += [f for f in v if f in df.columns]
all_feature_cols = list(dict.fromkeys(all_feature_cols))
print('Total features:', len(all_feature_cols))

## 4. Split **par puits** + normalisation (sans SMOTE → `scale_pos_weight`)

In [ ]:
X = df[all_feature_cols].copy().fillna(0); y = df['is_contaminated'].copy()
groups = df['county'].values if SPATIAL_BLOCKING else df['gm_well_id'].values
full_idx, test_idx = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(X, y, groups))
tr_rel, val_rel = next(GroupShuffleSplit(1, test_size=0.25, random_state=42).split(X.iloc[full_idx], y.iloc[full_idx], groups[full_idx]))
tr_idx, val_idx = full_idx[tr_rel], full_idx[val_rel]
X_train, X_val, X_test = X.iloc[tr_idx], X.iloc[val_idx], X.iloc[test_idx]
y_train = y.iloc[tr_idx].reset_index(drop=True); y_val = y.iloc[val_idx].reset_index(drop=True); y_test = y.iloc[test_idx].reset_index(drop=True)
print('overlap groupes tr/test =', len(set(groups[tr_idx]) & set(groups[test_idx])))
scaler = StandardScaler(); X_train_s = scaler.fit_transform(X_train); X_val_s = scaler.transform(X_val); X_test_s = scaler.transform(X_test)
scale_pos_weight = float((y_train==0).sum()/(y_train==1).sum())
print(f'Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | scale_pos_weight {scale_pos_weight:.3f}')

## 5. Graphe hétérogène — arêtes `sample↔sample` **spatiales** + **espace des features**

Relation de similarité entre puits : `near_geo` (KNN spatial sur lat/lon, **k=8**).
La relation `near_feat` (KNN dans l'espace des features) est **désactivée par défaut**
(`KNN_FEATURE_K=0`) : c'est elle qui densifiait trop le graphe en 06 (sur-lissage + RAM).
KMeans et KNN sont fittés *par split* (intra-split → pas de fuite).

In [ ]:
def fit_graph_transformers(X_train_df, n_geo_clusters=20):
    t = {}; coords = X_train_df[['latitude','longitude']].values.astype(np.float32)
    t['kmeans'] = KMeans(min(n_geo_clusters, max(2, len(X_train_df)//50)), random_state=42, n_init=10).fit(coords)
    return t
def knn_edges(M, k):
    k = min(k+1, len(M)); nn = NearestNeighbors(n_neighbors=k).fit(M); _, idx = nn.kneighbors(M)
    src = np.repeat(np.arange(len(M)), idx.shape[1]-1); dst = idx[:,1:].reshape(-1)
    e = np.stack([src, dst]); return np.concatenate([e, e[::-1]], 1)
def create_heterogeneous_graph(X_data, y_data, fc, transformers):
    d = HeteroData(); n = len(X_data)
    d['sample'].x = torch.FloatTensor(X_data.values); d['sample'].y = torch.LongTensor(np.asarray(y_data))
    ei = torch.stack([torch.arange(n), torch.arange(n)], 0)
    spec = {'Environmental':('env','has_env','affects'),'Facility_Proximity':('facility','near','near_to'),
            'Water_Quality':('water','quality','measured_in'),'Air_Quality':('air','breathes','over'),
            'Temporal':('time','sampled_at','at'),'Context':('context','described_by','describes')}
    for key,(nt,e1,e2) in spec.items():
        feats = [f for f in fc.get(key,[]) if f in X_data.columns]
        if feats:
            d[nt].x = torch.FloatTensor(X_data[feats].values); d['sample',e1,nt].edge_index = ei; d[nt,e2,'sample'].edge_index = ei
    km = transformers['kmeans']; coords = X_data[['latitude','longitude']].values.astype(np.float32); lab = km.predict(coords)
    d['geo_cluster'].x = torch.FloatTensor(km.cluster_centers_.astype(np.float32)); src = np.arange(n)
    d['sample','located_in','geo_cluster'].edge_index = torch.tensor([src,lab], dtype=torch.long)
    d['geo_cluster','contains','sample'].edge_index = torch.tensor([lab,src], dtype=torch.long)
    if KNN_SPATIAL_K and KNN_SPATIAL_K > 0:
        d['sample','near_geo','sample'].edge_index = torch.tensor(knn_edges(coords, KNN_SPATIAL_K), dtype=torch.long)
    if KNN_FEATURE_K and KNN_FEATURE_K > 0:   # désactivé par défaut (anti sur-lissage/RAM)
        d['sample','near_feat','sample'].edge_index = torch.tensor(knn_edges(X_data.values.astype(np.float32), KNN_FEATURE_K), dtype=torch.long)
    return d

transformers = fit_graph_transformers(pd.DataFrame(X_train_s, columns=X_train.columns))
train_data = create_heterogeneous_graph(pd.DataFrame(X_train_s, columns=X_train.columns), y_train, FEATURE_CATEGORIES, transformers)
val_data   = create_heterogeneous_graph(pd.DataFrame(X_val_s, columns=X_val.columns), y_val, FEATURE_CATEGORIES, transformers)
test_data  = create_heterogeneous_graph(pd.DataFrame(X_test_s, columns=X_test.columns), y_test, FEATURE_CATEGORIES, transformers)
print('Node types:', list(train_data.node_types))
ss_edges = sum(train_data[et].edge_index.shape[1] for et in train_data.edge_types if et[0]=='sample' and et[2]=='sample')
print('Edge types:', len(train_data.edge_types), '| arêtes sample<->sample:', ss_edges)

## 6. HGT (GPU) — modèle, entraînement, recherche Optuna

In [ ]:
class HGT_Classifier(nn.Module):
    # Résiduel + LayerNorm par couche/type de nœud -> limite le sur-lissage (cf. notebook 06).
    def __init__(self, metadata, hidden_channels=128, out_channels=2, num_heads=4, num_layers=2,
                 dropout=0.3, residual=True):
        super().__init__()
        self.residual = residual; self.node_types = list(metadata[0])
        self.lin_dict = nn.ModuleDict({nt: Linear(-1, hidden_channels) for nt in metadata[0]})
        self.convs = nn.ModuleList([HGTConv(hidden_channels, hidden_channels, metadata, heads=num_heads) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.ModuleDict({nt: nn.LayerNorm(hidden_channels) for nt in metadata[0]}) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(nn.Linear(hidden_channels, hidden_channels//2), nn.ReLU(),
                                        nn.Dropout(dropout), nn.Linear(max(1,hidden_channels//2), out_channels))
    def forward(self, x_dict, edge_index_dict, return_embeddings=False):
        x_dict = {nt: self.lin_dict[nt](x).relu_() for nt, x in x_dict.items()}
        for conv, norm in zip(self.convs, self.norms):
            h = conv(x_dict, edge_index_dict)
            if self.residual:   # x + conv(x) puis LayerNorm + ReLU + dropout
                x_dict = {nt: self.dropout(F.relu(norm[nt](h[nt] + x_dict[nt]))) if nt in x_dict else h[nt] for nt in h}
            else:
                x_dict = {nt: self.dropout(F.relu(norm[nt](h[nt]))) for nt in h}
        emb = x_dict['sample']
        return emb if return_embeddings else self.classifier(emb)

def to_device(data):
    return ({nt: data[nt].x.to(device) for nt in data.node_types},
            {et: data[et].edge_index.to(device) for et in data.edge_types})
def augment_hetero_graph(x_dict, edge_dict, noise=0.01, drop=0.1):
    xa = {nt: x + noise*torch.randn_like(x) for nt, x in x_dict.items()}; ea = {}
    for et, e in edge_dict.items():
        E = e.size(1); keep = (torch.rand(E, device=device) > drop).nonzero(as_tuple=True)[0]; ea[et] = e if keep.numel()==0 else e[:, keep]
    return xa, ea
def train_hgt(model, train_data, val_data, epochs, lr=3e-3, weight_decay=1e-5, verbose=True, patience=None):
    model = model.to(device); opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    cc = np.bincount(train_data['sample'].y.numpy()); cw = torch.FloatTensor([1.0/c for c in cc]).to(device); cw = cw/cw.sum()
    crit = nn.CrossEntropyLoss(weight=cw); sch = ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=12)
    txd, ted = to_device(train_data); ty = train_data['sample'].y.to(device)
    vxd, ved = to_device(val_data); vy = val_data['sample'].y
    best = 0; best_state = None; no_improve = 0; hist = {'train_loss':[], 'val_f1':[]}
    for ep in range(1, epochs+1):
        model.train(); opt.zero_grad(); xa, ea = augment_hetero_graph(txd, ted); out = model(xa, ea)
        loss = crit(out, ty); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): vp = model(vxd, ved).argmax(1).cpu(); vf1 = f1_score(vy, vp)
        sch.step(vf1); hist['train_loss'].append(loss.item()); hist['val_f1'].append(vf1)
        if vf1 > best + 1e-4:
            best = vf1; best_state = {k: v.clone() for k, v in model.state_dict().items()}; no_improve = 0
        else:
            no_improve += 1
        if verbose and (ep % 20 == 0 or ep == 1): print(f'Epoch {ep:3d}/{epochs} | Loss {loss:.4f} | Val F1 {vf1:.4f}')
        if patience and no_improve >= patience:
            if verbose: print(f'Early-stopping à epoch {ep} (pas d\'amélioration depuis {patience}).'); break
    if best_state: model.load_state_dict(best_state)
    if verbose: print(f'Meilleur Val F1: {best:.4f}')
    return model, hist

metadata = train_data.metadata()
t0 = time.time()
if RUN_OPTUNA_HGT:
    def objective(trial):
        m = HGT_Classifier(metadata,
            hidden_channels=trial.suggest_categorical('hidden_channels',[64,128]),   # RAM
            num_heads=trial.suggest_categorical('num_heads',[2,4]),
            num_layers=trial.suggest_int('num_layers',1,2),                          # <=2 (anti sur-lissage)
            dropout=trial.suggest_float('dropout',0.2,0.5),
            residual=RESIDUAL_HGT)
        m, h = train_hgt(m, train_data, val_data, epochs=HGT_OPTUNA_EPOCHS,
                         lr=trial.suggest_float('lr',1e-4,1e-2,log=True),
                         weight_decay=trial.suggest_float('weight_decay',1e-5,1e-3,log=True),
                         verbose=False, patience=EARLY_STOP_PATIENCE)
        return max(h['val_f1'])
    study_hgt = optuna.create_study(direction='maximize')
    study_hgt.optimize(objective, n_trials=HGT_OPTUNA_TRIALS, show_progress_bar=True)
    bp = study_hgt.best_params; print('Meilleurs params HGT:', bp, '| Val F1', round(study_hgt.best_value,4))
    hgt_model = HGT_Classifier(metadata, hidden_channels=bp['hidden_channels'], num_heads=bp['num_heads'],
                               num_layers=bp['num_layers'], dropout=bp['dropout'], residual=RESIDUAL_HGT)
    hgt_model, hgt_history = train_hgt(hgt_model, train_data, val_data, epochs=EPOCHS_HGT,
                                       lr=bp['lr'], weight_decay=bp['weight_decay'], patience=EARLY_STOP_PATIENCE)
else:
    study_hgt = None
    hgt_model = HGT_Classifier(metadata, hidden_channels=HIDDEN_DEFAULT, residual=RESIDUAL_HGT)
    hgt_model, hgt_history = train_hgt(hgt_model, train_data, val_data, epochs=EPOCHS_HGT, patience=EARLY_STOP_PATIENCE)
print(f'Temps HGT: {(time.time()-t0)/60:.1f} min')

ep = range(1, len(hgt_history['train_loss'])+1)
fig, ax1 = plt.subplots(figsize=(8,5)); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Train loss', color='tab:blue')
l1 = ax1.plot(ep, hgt_history['train_loss'], color='tab:blue', lw=2, label='Train loss'); ax1.grid(alpha=0.3); ax1.set_ylim(bottom=0)
ax2 = ax1.twinx(); ax2.set_ylabel('Val F1', color='tab:orange'); ax2.set_ylim([0,1.05])
l2 = ax2.plot(ep, hgt_history['val_f1'], color='tab:orange', lw=2, label='Validation F1')
ax1.legend(l1+l2, [x.get_label() for x in l1+l2], loc='center right'); fig.tight_layout()
plt.savefig(OUT_FIG/'fig3_hgt_learning_curve.png', dpi=300, bbox_inches='tight'); plt.show()

## 7. Modèles de base (GPU) : XGBoost + LightGBM + CatBoost

In [ ]:
def make_xgb(**kw):
    p = dict(random_state=42, eval_metric='logloss', tree_method='hist', device=XGB_DEVICE, scale_pos_weight=scale_pos_weight)
    p.update(kw); return xgb.XGBClassifier(**p)
xgb_baseline = make_xgb(n_estimators=400, max_depth=6, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8)
xgb_baseline.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=False)
lgb_model = lgb.LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, num_leaves=31, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, scale_pos_weight=scale_pos_weight, random_state=42,
    verbose=-1, device='gpu' if GPU else 'cpu')
try:
    lgb_model.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], callbacks=[lgb.log_evaluation(0)])
except Exception as e:
    print('LightGBM GPU indisponible -> CPU', e); lgb_model = lgb.LGBMClassifier(n_estimators=500, max_depth=6,
        learning_rate=0.05, num_leaves=31, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
        scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1)
    lgb_model.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], callbacks=[lgb.log_evaluation(0)])
if ADD_CATBOOST:
    cb_model = CatBoostClassifier(iterations=600, depth=6, learning_rate=0.05, scale_pos_weight=scale_pos_weight,
        random_state=42, verbose=0, task_type=CB_TASK)
    cb_model.fit(X_train_s, y_train)
else:
    cb_model = None
print('XGB F1', round(f1_score(y_test, xgb_baseline.predict(X_test_s)),4),
      '| LGB F1', round(f1_score(y_test, lgb_model.predict(X_test_s)),4),
      ('| CB F1 '+str(round(f1_score(y_test, cb_model.predict(X_test_s).astype(int)),4))) if cb_model is not None else '')

## 8. Helpers (embeddings / probas HGT, seuil optimal)

In [ ]:
def hgt_embeddings(model, data):
    model.eval(); xd, ed = to_device(data)
    with torch.no_grad(): return model(xd, ed, return_embeddings=True).cpu().numpy()
def hgt_proba(model, data):
    model.eval(); xd, ed = to_device(data)
    with torch.no_grad(): return F.softmax(model(xd, ed), 1).cpu().numpy()
def best_threshold(y_val, proba_val):
    ts = np.linspace(0.1, 0.9, 81)
    return float(ts[int(np.argmax([f1_score(y_val, (proba_val>=t).astype(int)) for t in ts]))])
train_emb = hgt_embeddings(hgt_model, train_data); val_emb = hgt_embeddings(hgt_model, val_data); test_emb = hgt_embeddings(hgt_model, test_data)
hgt_p_val = hgt_proba(hgt_model, val_data)[:,1]; hgt_p_test = hgt_proba(hgt_model, test_data)[:,1]

## 9. APPROCHE 1 — Embedding Fusion (HGT → PCA → XGBoost, Optuna)

In [ ]:
pca_emb = PCA(0.95, random_state=42)
train_emb_pca = pca_emb.fit_transform(train_emb); val_emb_pca = pca_emb.transform(val_emb); test_emb_pca = pca_emb.transform(test_emb)
X_train_fusion = np.concatenate([X_train_s, train_emb_pca], 1)
X_val_fusion   = np.concatenate([X_val_s,   val_emb_pca],   1)
X_test_fusion  = np.concatenate([X_test_s,  test_emb_pca],  1)
print('Fusion dims:', X_train_fusion.shape[1], '| PCA emb:', train_emb_pca.shape[1])
def objective_fusion(t):
    c = make_xgb(n_estimators=t.suggest_int('n_estimators',200,800), max_depth=t.suggest_int('max_depth',3,12),
        learning_rate=t.suggest_float('learning_rate',0.01,0.3,log=True), subsample=t.suggest_float('subsample',0.5,1.0),
        colsample_bytree=t.suggest_float('colsample_bytree',0.4,1.0), min_child_weight=t.suggest_int('min_child_weight',1,10),
        gamma=t.suggest_float('gamma',0,1.0), reg_alpha=t.suggest_float('reg_alpha',1e-4,1.0,log=True),
        reg_lambda=t.suggest_float('reg_lambda',1e-4,2.0,log=True))
    c.fit(X_train_fusion, y_train, eval_set=[(X_val_fusion, y_val)], verbose=False)
    return f1_score(y_val, c.predict(X_val_fusion))
study_fusion = optuna.create_study(direction='maximize'); study_fusion.optimize(objective_fusion, n_trials=N_TRIALS, show_progress_bar=True)
xgb_fusion = make_xgb(**study_fusion.best_params); xgb_fusion.fit(X_train_fusion, y_train, eval_set=[(X_val_fusion, y_val)], verbose=False)
fusion_p_val = xgb_fusion.predict_proba(X_val_fusion)[:,1]; fusion_p_test = xgb_fusion.predict_proba(X_test_fusion)[:,1]
print('Fusion F1@0.5', round(f1_score(y_test,(fusion_p_test>=0.5).astype(int)),4), '| AUC', round(roc_auc_score(y_test,fusion_p_test),4))

## 10. APPROCHE 2 — Stacking out-of-fold (HGT+XGB+LGB+CatBoost) + calibration

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=42)
xgb_oof = cross_val_predict(make_xgb(n_estimators=400, max_depth=6, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8),
                            X_train_s, y_train, cv=skf, method='predict_proba')
lgb_oof = cross_val_predict(lgb.LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, num_leaves=31, subsample=0.8,
                            colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1),
                            X_train_s, y_train, cv=skf, method='predict_proba')
hgt_p_tr = hgt_proba(hgt_model, train_data); hgt_p_va = hgt_proba(hgt_model, val_data); hgt_p_te = hgt_proba(hgt_model, test_data)
xgb_p_va, xgb_p_te = xgb_baseline.predict_proba(X_val_s), xgb_baseline.predict_proba(X_test_s)
lgb_p_va, lgb_p_te = lgb_model.predict_proba(X_val_s),   lgb_model.predict_proba(X_test_s)
if cb_model is not None:
    cb_oof = cross_val_predict(CatBoostClassifier(iterations=600, depth=6, learning_rate=0.05, scale_pos_weight=scale_pos_weight, random_state=42, verbose=0),
                               X_train_s, y_train, cv=skf, method='predict_proba')
    cb_p_va, cb_p_te = cb_model.predict_proba(X_val_s), cb_model.predict_proba(X_test_s)
else:
    cb_oof = cb_p_va = cb_p_te = None
pca_meta = PCA(0.90, random_state=42); etr = pca_meta.fit_transform(train_emb); eva = pca_meta.transform(val_emb); ete = pca_meta.transform(test_emb)
def build_meta(h, x, l, c, e):
    eps = 1e-10; h1, x1, l1 = h[:,1:2], x[:,1:2], l[:,1:2]
    parts = [h1, x1, l1, (-h*np.log(h+eps)).sum(1,keepdims=True), (-x*np.log(x+eps)).sum(1,keepdims=True), (-l*np.log(l+eps)).sum(1,keepdims=True),
             h.max(1,keepdims=True), x.max(1,keepdims=True), l.max(1,keepdims=True),
             np.abs(h1-x1), np.abs(x1-l1), h1*x1, np.maximum(x1,l1), 0.3*h1+0.4*x1+0.3*l1]
    if c is not None:
        c1 = c[:,1:2]; parts += [c1, (-c*np.log(c+eps)).sum(1,keepdims=True), np.abs(x1-c1), 0.25*(h1+x1+l1+c1)]
    parts.append(e); return np.hstack(parts)
meta_train = build_meta(hgt_p_tr, xgb_oof, lgb_oof, cb_oof, etr)
meta_val   = build_meta(hgt_p_va, xgb_p_va, lgb_p_va, cb_p_va, eva)
meta_test  = build_meta(hgt_p_te, xgb_p_te, lgb_p_te, cb_p_te, ete)
print('Méta-features:', meta_train.shape[1])
def objective_stacking(t):
    c = make_xgb(n_estimators=t.suggest_int('n_estimators',50,400), max_depth=t.suggest_int('max_depth',2,8),
        learning_rate=t.suggest_float('learning_rate',0.01,0.2,log=True), subsample=t.suggest_float('subsample',0.5,1.0),
        colsample_bytree=t.suggest_float('colsample_bytree',0.5,1.0), min_child_weight=t.suggest_int('min_child_weight',1,8),
        reg_alpha=t.suggest_float('reg_alpha',1e-4,0.5,log=True), reg_lambda=t.suggest_float('reg_lambda',1e-4,1.0,log=True))
    c.fit(meta_train, y_train, eval_set=[(meta_val, y_val)], verbose=False)
    return f1_score(y_val, c.predict(meta_val))
study_stack = optuna.create_study(direction='maximize'); study_stack.optimize(objective_stacking, n_trials=N_TRIALS, show_progress_bar=True)
meta_clf = make_xgb(**study_stack.best_params); meta_clf.fit(meta_train, y_train, eval_set=[(meta_val, y_val)], verbose=False)
stack_p_val = meta_clf.predict_proba(meta_val)[:,1]; stack_p_test = meta_clf.predict_proba(meta_test)[:,1]
iso = IsotonicRegression(out_of_bounds='clip').fit(stack_p_val, y_val)
stack_p_val_cal = iso.transform(stack_p_val); stack_p_test_cal = iso.transform(stack_p_test)
print('Stacking F1@0.5', round(f1_score(y_test,(stack_p_test>=0.5).astype(int)),4),
      '| Brier brut', round(brier_score_loss(y_test, stack_p_test),4),
      '| Brier calibré', round(brier_score_loss(y_test, stack_p_test_cal),4))

## 11. Comparaison finale + figures

In [ ]:
models = [('HGT', hgt_p_val, hgt_p_test),
          ('XGBoost', xgb_p_va[:,1], xgb_p_te[:,1]),
          ('LightGBM', lgb_p_va[:,1], lgb_p_te[:,1])]
if cb_model is not None: models.append(('CatBoost', cb_p_va[:,1], cb_p_te[:,1]))
models += [('Embedding Fusion', fusion_p_val, fusion_p_test),
           ('Stacking', stack_p_val, stack_p_test),
           ('Stacking+calib', stack_p_val_cal, stack_p_test_cal)]
rows = []
for name, pv, pte in models:
    thr = best_threshold(y_val, pv); ypt = (pte>=thr).astype(int)
    rows.append({'Model':name, 'thr':round(thr,3), 'F1@0.5':f1_score(y_test,(pte>=0.5).astype(int)), 'F1@thr':f1_score(y_test,ypt),
                 'Precision@thr':precision_score(y_test,ypt), 'Recall@thr':recall_score(y_test,ypt),
                 'ROC_AUC':roc_auc_score(y_test,pte), 'AP':average_precision_score(y_test,pte), 'Brier':brier_score_loss(y_test,pte)})
results = pd.DataFrame(rows); print(results.to_string(index=False))

fig, axes = plt.subplots(1,2,figsize=(15,5)); cols = plt.cm.viridis(np.linspace(0.1,0.85,len(results)))
for ax, metric in zip(axes, ['F1@thr','ROC_AUC']):
    bb = ax.bar(results['Model'], results[metric], color=cols, edgecolor='black')
    ax.set_ylabel(metric, fontweight='bold'); ax.set_ylim([0, max(0.95, results[metric].max()*1.08)]); ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=20, labelsize=8)
    for b in bb: ax.text(b.get_x()+b.get_width()/2., b.get_height(), f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
plt.tight_layout(); plt.savefig(OUT_FIG/'fig4_models_comparison.png', dpi=300, bbox_inches='tight'); plt.show()

plt.figure(figsize=(8,6))
for name, _, pte in models:
    if name=='Stacking+calib': continue
    fpr, tpr, _ = roc_curve(y_test, pte); plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC={roc_auc_score(y_test,pte):.3f})')
plt.plot([0,1],[0,1],'k--',alpha=0.5); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_FIG/'fig5_roc_curves.png', dpi=300, bbox_inches='tight'); plt.show()

plt.figure(figsize=(7,6)); plt.plot([0,1],[0,1],'k--',label='Parfait')
for proba, lab, mk in [(stack_p_test,'Stacking brut','o-'),(stack_p_test_cal,'Stacking calibré','s-')]:
    pt, pp = calibration_curve(y_test, proba, n_bins=10); plt.plot(pp, pt, mk, lw=2, label=lab)
plt.xlabel('Probabilité prédite'); plt.ylabel('Fraction positive observée'); plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_FIG/'fig_calibration_stacking.png', dpi=300, bbox_inches='tight'); plt.show()
print('Meilleur modèle (F1@thr):', results.loc[results['F1@thr'].idxmax(),'Model'])

## 12. Sauvegarde (à renvoyer)

In [ ]:
torch.save({'model_state_dict': hgt_model.state_dict(), 'metadata': train_data.metadata()}, OUT_MODELS/'hgt_v2.pt')
with open(OUT_MODELS/'xgb_fusion_v2.pkl','wb') as f: pickle.dump(xgb_fusion, f)
with open(OUT_MODELS/'meta_clf_v2.pkl','wb') as f: pickle.dump(meta_clf, f)
with open(OUT_MODELS/'scaler_v2.pkl','wb') as f: pickle.dump(scaler, f)
results.to_csv(OUT_RES/'results_v2.csv', index=False)
summary = {'target':'is_contaminated (EPA 2024)', 'device':str(device), 'n_features':len(all_feature_cols),
           'split':'group by '+('county' if SPATIAL_BLOCKING else 'gm_well_id'),
           'config':{'EPOCHS_HGT':EPOCHS_HGT, 'EARLY_STOP_PATIENCE':EARLY_STOP_PATIENCE, 'RUN_OPTUNA_HGT':RUN_OPTUNA_HGT,
                     'HGT_OPTUNA_TRIALS':HGT_OPTUNA_TRIALS, 'N_TRIALS':N_TRIALS, 'KNN_SPATIAL_K':KNN_SPATIAL_K,
                     'KNN_FEATURE_K':KNN_FEATURE_K, 'RESIDUAL_HGT':RESIDUAL_HGT, 'catboost':bool(cb_model is not None)},
           'best_hgt_params': (study_hgt.best_params if RUN_OPTUNA_HGT else None),
           'hgt_epochs_run': len(hgt_history['train_loss']),
           'n_train':int(len(y_train)),'n_val':int(len(y_val)),'n_test':int(len(y_test)),
           'results':results.to_dict('records'), 'best_model':results.loc[results['F1@thr'].idxmax(),'Model']}
with open(OUT_RES/'summary_v2.json','w') as f: json.dump(summary, f, indent=2, default=float)
print('Sauvegardé dans outputs/hgt_hybrid_v2/ et models/. Renvoyer results_v2.csv + summary_v2.json')

## Lecture des résultats attendue

Si la correction fonctionne, on doit voir, vs notebook 06 : **HGT et surtout Embedding
Fusion remonter** (objectif Fusion AUC ≥ 0.91 / F1 ≥ 0.81 comme le notebook 05), pendant
que XGB/LGBM/CatBoost restent ~stables (ils n'utilisent pas le graphe). L'early-stopping
doit couper bien avant 200 epochs. Comparer `summary_v2.json` à `outputs/hgt_hybrid_improved/`.

## Pistes supplémentaires (si le budget GPU le permet)
1. **GATv2 / SAGE** via `HeteroConv` (comparer à HGT résiduel).
2. **Validation spatiale par blocs** (`SPATIAL_BLOCKING=True`) + répétition sur graines → barres d'incertitude.
3. **Mini-batch graphe** (`NeighborLoader`) pour entraîner plus longtemps sans saturer la VRAM.
4. **Seuil coût-sensible** (faux négatif ≫ faux positif) plutôt que F1 brut.
5. **Multi-tâches** : régression auxiliaire `log(sum_pfas)` pour enrichir les embeddings.
6. **Ensembling de graines** (5 HGT) pour réduire la variance des embeddings.